## BT4012 Kaggle Competition ##
Name: Zhong Wanzhi <br>

In [1]:
import pandas as pd
import numpy as np

## Import the respective datasets ##

In [2]:
from pathlib import Path

DATA_DIR = Path("data")

edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
train = pd.read_csv(DATA_DIR / "train.csv")

## Graph-Based Feature Engineering

I added graph-based features because Bitcoin transactions are connected through payment flows where an output from one transaction becomes an input to another. These connections capture each transaction’s position in the network and can reflect patterns such as consolidating deposits or distributing payments.

A directed graph is hence constructed using txs_edgelist.csv where txId1 -> txId2 indicates that an output of txId1 is consumed by txId2.
Five additional features are calculated:

1. In-degree: Number of distinct transactions pointing to the transaction.

2. Out-degree: Number of distinct subsequent transactions it points to.

3. Total degree: Sum of incoming and outgoing connections.

4. PageRank: A measure of network importance -> a transaction receives a higher score when transactions pointing to it have high scores, with each contributing according to its number of outgoing connections.

5. Approximate betweenness centrality: Estimates how often a transaction lies on the shortest directed paths between other transactions. A higher score suggests that it acts as a bridge in the network. To reduce computation time, I sample up to 256 source nodes and use a fixed random seed of 42 for reproducibility.

I also create 30 neighbour-average features by averaging feat_1 to feat_30 across each transaction’s immediate incoming and outgoing neighbours. Only feature values from labelled training transactions are included in these averages. Transactions with no eligible neighbours receive zero values.

In [3]:
import networkx as nx

# Build directed graph: txId1 -> txId2
G = nx.from_pandas_edgelist(
    edges,
    source="txId1",
    target="txId2",
    create_using=nx.DiGraph(),
)

# Additional graph features
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
pagerank = nx.pagerank(G, alpha=0.85)
betweenness_samples = min(256, G.number_of_nodes())

print(
    f"Calculating approximate betweenness using "
    f"{betweenness_samples} sampled source nodes..."
)

betweenness = (
    nx.betweenness_centrality(
        G,
        k=betweenness_samples,
        normalized=True,
        endpoints=False,
        seed=42,
    )
    if G.number_of_nodes() > 2
    else dict.fromkeys(G.nodes, 0.0)
)

graph_cols = [
    "in_degree",
    "out_degree",
    "total_degree",
    "pagerank",
    "betweenness",
]

# Apply the same graph features to train and test.
for df in (train, test):
    df["in_degree"] = df["txId"].map(in_degrees).fillna(0)
    df["out_degree"] = df["txId"].map(out_degrees).fillna(0)
    df["total_degree"] = df["in_degree"] + df["out_degree"]
    df["pagerank"] = df["txId"].map(pagerank).fillna(0)
    df["betweenness"] = df["txId"].map(betweenness).fillna(0)

# Existing neighbour aggregation
# use both incoming and outgoing neighbours
G_undirected = G.to_undirected()

neighbor_source_cols = [f"feat_{i}" for i in range(1, 31)]
neighbor_cols = [f"neighbor_avg_feat_{i}" for i in range(1, 31)]

# use feature values from labelled training transactions only
train_labeled = train.loc[train["label"].isin([0, 1])]

node_features = dict(
    zip(
        train_labeled["txId"],
        train_labeled[neighbor_source_cols].to_numpy(),
    )
)


def aggregate_neighbor_features(node_ids):
    """Average 30 features across eligible immediate neighbours."""
    aggregated = np.zeros(
        (len(node_ids), len(neighbor_source_cols))
    )

    for row_index, node_id in enumerate(node_ids):
        if node_id not in G_undirected:
            continue

        neighbor_values = [
            node_features[neighbor]
            for neighbor in G_undirected.neighbors(node_id)
            if neighbor in node_features
        ]

        if neighbor_values:
            aggregated[row_index] = np.mean(
                neighbor_values, axis=0
            )

    return aggregated


# Use the same training-node lookup for both datasets
train[neighbor_cols] = aggregate_neighbor_features(train["txId"])
test[neighbor_cols] = aggregate_neighbor_features(test["txId"])

feature_cols = (
    [f"feat_{i}" for i in range(1, 166)]
    + graph_cols
    + neighbor_cols
)

print(f"Feature engineering complete: {len(feature_cols)} features.")

Calculating approximate betweenness using 256 sampled source nodes...
Feature engineering complete: 200 features.


## Class Balancing with SMOTE

To handle missing values consistently just in case there are any, I used training medians for both datasets, preventing test data from influencing the preprocessing. Zero served as a fallback when a feature had no available values to calculate a median.

After handling missing values, I standardized the features in preparation for SMOTE. Since SMOTE uses distances to find similar samples, scaling prevents features with larger numerical ranges from dominating these distances.

I then used SMOTE to address class imbalance and help the model learn minority-class patterns. It was applied only to the training data so that the test data retained its original distribution.

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# All labeled training data, using original + graph + neighbor features
labeled_mask = train["label"].isin([0, 1])

X_train = train.loc[labeled_mask, feature_cols].copy() # Making a copy of the training features
y_train = train.loc[labeled_mask, "label"].astype(int) # Making a copy of the training labels
X_test = test[feature_cols].copy() # Making a copy of the test features

# Fill missing values using training column medians or 0 if a column has no values
medians = X_train.median().fillna(0)
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

# Scale before SMOTE because neighbor distances depend on feature scales
# Fit preprocessing on the real training rows only
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=feature_cols, index=X_test.index
)

# Generate synthetic examples for training only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled, y_train
)
print("Before SMOTE:", y_train.value_counts().sort_index().to_dict())
print("After SMOTE:", y_train_resampled.value_counts().sort_index().to_dict())

Before SMOTE: {0: 27591, 1: 3644}
After SMOTE: {0: 27591, 1: 27591}


## Model

I chose HistGradientBoostingClassifier because it can capture nonlinear relationships and interactions between the original, graph and neighbor features.  Among the models evaluated, it achieved the highest precision while maintaining a comparable ROC-AUC.

The model uses 200 boosting iterations with a learning rate of 0.05 to build predictions gradually. A maximum tree depth of 7 limits model complexity. Since SMOTE already balances the training data, no additional class weighting is applied. Early stopping is disabled to run all 200 iterations and a fixed random seed supports reproducibility.

The predicted probability of class 1 for each test transaction is saved in the required submission format.

In [ ]:
model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05, 
    max_depth=7,
    class_weight=None,     # Training data is already balanced with SMOTE.
    early_stopping=False, 
    random_state=42,
)

model.fit(X_train_resampled, y_train_resampled)

# Probability of class 1, matching your original submission format.
test_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

submission = pd.DataFrame({
    "index": test["index"].to_numpy(),
    "target": test_pred_proba,
})

submission.to_csv("final_submission.csv", index=False)

print("Saved final_submission.csv")

Saved testing_submission.csv
